# 03 — Full Pipeline: Baseline → Dataset → GRPO Training

Flusso completo di sviluppo su **Google Colab** con **Unsloth**:

1. **Setup & Imports** — installa dipendenze, verifica GPU
2. **Genera Dataset** — crea il dataset sintetico (5000 samples)
3. **Baseline Evaluation** — valuta il modello base (senza fine-tuning) e logga su wandb
4. **GRPO Training** — addestra il modello con GRPO e logga su wandb
5. **Post-Training Evaluation** — valuta il modello addestrato e confronta con baseline

> **Requisiti:** GPU su Colab (T4 minimo, A100 ideale)

## 1. Setup su Colab

In [1]:
!rm -rf tris
!git clone https://github.com/GiuseppeBellamacina/tris.git
%cd tris
!bash setup_linux.sh

Cloning into 'tris'...
remote: Enumerating objects: 213, done.
remote: Counting objects: 100% (213/213), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 213 (delta 105), reused 177 (delta 69), pack-reused 0 (from 0)
Receiving objects: 100% (213/213), 541.14 KiB | 13.20 MiB/s, done.
Resolving deltas: 100% (105/105), done.
/content/tris
=== Setup GRPO Strict Generation (Colab) ===

📦 Installazione uv...
✅ uv installato

📦 Installazione dipendenze + progetto...
Using Python 3.12.13 environment at: /usr
Resolved 282 packages in 805ms                                       
Prepared 1 package in 346ms                                              
Uninstalled 1 package in 0.50ms
Installed 1 package in 0.98ms==0.1.0 (from file:///content/t
 ~ grpo-strict-generation==0.1.0 (from file:///content/tris)
✅ Dipendenze installate + progetto in editable mode

🔍 Verifica installazione...

AMBIENTE GRPO STRICT GENERATION

🔥 PyTorch:
   Version: 2.10.0+cu128
   CUDA available: True

## 2. Imports & GPU Check

In [3]:
from unsloth import FastLanguageModel

print("Unsloth importato correttamente")

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: Nessuna GPU trovata!")

import transformers
import trl

print(f"transformers: {transformers.__version__}")
print(f"trl:          {trl.__version__}")

Unsloth importato correttamente
PyTorch version: 2.10.0+cu128
CUDA available:  True
GPU:             Tesla T4
VRAM:            15.6 GB
transformers: 4.57.6
trl:          0.24.0


In [4]:
import wandb

# Login a wandb — la prima volta su Colab chiede l'API key.
# Prendi la tua key da https://wandb.ai/authorize
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: cosmico445 (cosmico445-cosmic-inc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
import os
from pathlib import Path

ROOT = Path.cwd()
os.chdir(ROOT)

# Costanti
WANDB_PROJECT = "grpo-strict-generation"
DATASET_PATH = ROOT / "data" / "synthetic"
CONFIG_PATH = ROOT / "experiments" / "configs" / "grpo.yaml"

print(f"Root progetto: {ROOT}")

Root progetto: /content/tris


## 3. Genera Dataset Sintetico

Generiamo un dataset serio con **5000 samples** (3 livelli di difficoltà: simple, medium, hard).

In [6]:
from src.datasets.dataloader import load_synthetic_dataset
from src.datasets.synthetic_dataset import generate_dataset

NUM_SAMPLES = 5000
SEED = 42

if DATASET_PATH.exists():
    print(f"Dataset già presente in {DATASET_PATH}, lo ricarico...")
else:
    print(f"Genero {NUM_SAMPLES} samples (seed={SEED})...")
    ds = generate_dataset(num_samples=NUM_SAMPLES, seed=SEED)
    ds.save_to_disk(str(DATASET_PATH))
    print(f"Dataset salvato in {DATASET_PATH}")

ds = load_synthetic_dataset(str(DATASET_PATH))

for split_name, split_ds in ds.items():
    print(f"  {split_name}: {len(split_ds)} samples")
    # Distribuzione per difficoltà
    diffs = split_ds["difficulty"]
    for d in ["simple", "medium", "hard"]:
        count = sum(1 for x in diffs if x == d)
        print(f"    json/{d}: {count}")

Genero 5000 samples (seed=42)...


Saving the dataset (0/1 shards):   0%|          | 0/4000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset salvato in /content/tris/data/synthetic
  train: 4000 samples
    json/simple: 1606
    json/medium: 1417
    json/hard: 977
  test: 1000 samples
    json/simple: 413
    json/medium: 337
    json/hard: 250


## 4. Baseline Evaluation

Valutiamo il modello **base** (senza fine-tuning) sul test set.
Questo ci dà il punto di partenza per misurare il miglioramento dopo GRPO.

In [7]:
from src.datasets.dataloader import format_prompt_for_model
from src.evaluation.baseline_eval import generate_completions
from src.evaluation.evaluate import compute_detailed_metrics, pass_at_k
from src.models.model_loader import load_model_and_tokenizer
from src.utils.config import load_config

config = load_config(str(CONFIG_PATH))

# Carica modello BASE (senza LoRA) per la baseline
base_config = {"model": config["model"]}  # no "lora" key → no adapters
print(f"Caricamento modello base: {config['model']['name']}")
model, tokenizer = load_model_and_tokenizer(base_config)

ModuleNotFoundError: No module named 'src.utils.config'

In [ ]:
# Prepara prompt dal test set
test_ds = ds["test"]
MAX_EVAL_SAMPLES = 200  # limita per velocità su Colab
eval_ds = test_ds.select(range(min(MAX_EVAL_SAMPLES, len(test_ds))))

prompts = [format_prompt_for_model(eval_ds[i], tokenizer) for i in range(len(eval_ds))]
task_types = list(eval_ds["task_type"])
difficulties = list(eval_ds["difficulty"])

print(f"Prompts preparati: {len(prompts)}")

# Genera completions
gen_config = {
    "max_new_tokens": 512,
    "temperature": 0.7,
    "top_p": 0.95,
    "do_sample": True,
}

print("Generazione completions baseline...")
baseline_completions = generate_completions(
    model=model,
    tokenizer=tokenizer,
    prompts=prompts,
    generation_config=gen_config,
    num_return_sequences=1,
    batch_size=4,
)
print(f"Completions generate: {len(baseline_completions)}")

In [ ]:
# Calcola metriche baseline
first_completions = [comps[0] for comps in baseline_completions]
baseline_metrics = compute_detailed_metrics(first_completions, task_types, difficulties)

# Pass@k
baseline_pass_k = pass_at_k(baseline_completions, task_types, k_values=[1])

print(f"\n{'='*50}")
print(f"BASELINE — {config['model']['name']}")
print(f"{'='*50}")
print(f"Overall Pass@1: {baseline_metrics['overall_pass_rate']:.4f}")
print("\nPer-category breakdown:")
for cat, stats in baseline_metrics["per_category"].items():
    print(f"  {cat}: {stats['pass_rate']:.4f} ({stats['valid']}/{stats['total']})")

# Log su wandb
wandb.init(
    project=WANDB_PROJECT,
    name=f"baseline-{config['model']['name'].split('/')[-1]}",
    tags=["baseline", "notebook-03"],
    config={"model": config["model"], "eval_samples": len(prompts)},
)
wandb_metrics = {
    "overall_pass_rate": baseline_metrics["overall_pass_rate"],
    **baseline_pass_k,
}
for cat, stats in baseline_metrics["per_category"].items():
    wandb_metrics[f"pass_rate/{cat}"] = stats["pass_rate"]
wandb.log(wandb_metrics)
wandb.finish()
print("\nBaseline loggata su wandb")

In [ ]:
# Qualche esempio di completions baseline
from src.training.rewards import combined_reward

print("Esempi di completions baseline:\n")
for i in [0, 1, 5, 10]:
    if i >= len(eval_ds):
        break
    comp = first_completions[i]
    r = combined_reward(comp, task_type="json")
    print(f"--- [{difficulties[i]}] reward={r} ---")
    print(f"Prompt:     {eval_ds[i]['prompt'][:100]}...")
    print(f"Completion: {comp[:200]}")
    print()

In [ ]:
# Libera VRAM dal modello base
del model
torch.cuda.empty_cache()
print("VRAM liberata")

## 5. GRPO Training

Addestriamo il modello con **GRPO** (Group Relative Policy Optimization).
Il config viene letto da `experiments/configs/grpo.yaml`.

Parametri chiave:
- **learning_rate**: 5e-6
- **optim**: paged_adamw_8bit (meno VRAM)
- **max_grad_norm**: 0.1 (gradient clipping aggressivo)
- **num_generations**: 8 (gruppo per calcolare i vantaggi relativi)
- **beta**: 0.04 (KL penalty)

In [ ]:
from datasets import Dataset
from src.training.rewards import build_reward_function

# Ricarica modello CON LoRA per il training
print(f"Caricamento modello con LoRA: {config['model']['name']}")
model, tokenizer = load_model_and_tokenizer(config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parametri totali:    {total:,}")
print(f"Parametri trainable: {trainable:,} ({100 * trainable / total:.2f}%)")

In [ ]:
# Prepara dataset di training (prompt formattati)
train_ds = ds["train"]
formatted = []
for i in range(len(train_ds)):
    s = train_ds[i]
    p = format_prompt_for_model(s, tokenizer)
    formatted.append({"prompt": p})

prompt_dataset = Dataset.from_list(formatted)
print(f"Training dataset: {len(prompt_dataset)} prompt pronti")

# Reward function
reward_fn = build_reward_function(config["reward"])
print("Reward function creata")

In [ ]:
import datetime

from trl import GRPOConfig

training_cfg = config["training"]
grpo_cfg = config["grpo"]

output_dir = str(ROOT / "experiments" / "checkpoints" / "grpo_full")
log_dir = str(ROOT / "experiments" / "logs" / "grpo_full")
Path(output_dir).mkdir(parents=True, exist_ok=True)
Path(log_dir).mkdir(parents=True, exist_ok=True)

_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"grpo-full-{_ts}"

# bf16 vs fp16 in base alla GPU
_supports_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
use_bf16 = _supports_bf16
use_fp16 = not _supports_bf16 and torch.cuda.is_available()

grpo_config = GRPOConfig(
    output_dir=output_dir,
    run_name=run_name,
    # Training
    max_steps=training_cfg.get("max_steps", 1000),
    per_device_train_batch_size=training_cfg.get("per_device_train_batch_size", 1),
    gradient_accumulation_steps=training_cfg.get("gradient_accumulation_steps", 8),
    learning_rate=training_cfg.get("learning_rate", 5e-6),
    lr_scheduler_type=training_cfg.get("lr_scheduler_type", "cosine"),
    warmup_ratio=training_cfg.get("warmup_ratio", 0.1),
    optim=training_cfg.get("optim", "paged_adamw_8bit"),
    max_grad_norm=training_cfg.get("max_grad_norm", 0.1),
    bf16=use_bf16,
    fp16=use_fp16,
    # GRPO
    num_generations=grpo_cfg.get("num_generations", 8),
    max_completion_length=grpo_cfg.get("max_completion_length", 512),
    max_prompt_length=grpo_cfg.get("max_prompt_length", 256),
    beta=grpo_cfg.get("beta", 0.04),
    temperature=grpo_cfg.get("temperature", 0.7),
    # Logging
    logging_dir=log_dir,
    logging_steps=training_cfg.get("logging_steps", 10),
    save_steps=training_cfg.get("save_steps", 200),
    save_total_limit=training_cfg.get("save_total_limit", 3),
    report_to="wandb",
)

print("GRPOConfig creata")
print(f"  run_name:       {grpo_config.run_name}")
print(f"  max_steps:      {grpo_config.max_steps}")
print(f"  num_generations: {grpo_config.num_generations}")
print(f"  batch_size:     {grpo_config.per_device_train_batch_size}")
print(f"  grad_accum:     {grpo_config.gradient_accumulation_steps}")
print(f"  learning_rate:  {grpo_config.learning_rate}")
print(f"  optim:          {grpo_config.optim}")
print(f"  beta:           {grpo_config.beta}")
print(f"  bf16={use_bf16}, fp16={use_fp16}")

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=prompt_dataset,
    reward_funcs=reward_fn,
    processing_class=tokenizer,
)

print(f"Trainer inizializzato (tipo: {type(trainer).__name__})")
print(f"\nAvvio GRPO training ({grpo_config.max_steps} step)...\n")

trainer.train()
print("\nTraining completato!")

In [ ]:
# Salva il modello finale
final_path = Path(output_dir) / "final"
trainer.save_model(str(final_path))
tokenizer.save_pretrained(str(final_path))
print(f"Modello salvato in {final_path}")

## 6. Post-Training Evaluation

Valutiamo il modello addestrato sullo **stesso test set** usato per la baseline.

In [ ]:
# Genera completions con il modello trainato
# Usiamo gli stessi prompt della baseline per un confronto diretto
print("Generazione completions post-GRPO...")
grpo_completions = generate_completions(
    model=model,
    tokenizer=tokenizer,
    prompts=prompts,
    generation_config=gen_config,
    num_return_sequences=1,
    batch_size=4,
)

# Calcola metriche post-GRPO
grpo_first = [comps[0] for comps in grpo_completions]
grpo_metrics = compute_detailed_metrics(grpo_first, task_types, difficulties)
grpo_pass_k = pass_at_k(grpo_completions, task_types, k_values=[1])

print(f"\n{'='*50}")
print(f"POST-GRPO — {config['model']['name']}")
print(f"{'='*50}")
print(f"Overall Pass@1: {grpo_metrics['overall_pass_rate']:.4f}")
print("\nPer-category breakdown:")
for cat, stats in grpo_metrics["per_category"].items():
    print(f"  {cat}: {stats['pass_rate']:.4f} ({stats['valid']}/{stats['total']})")

# Log su wandb
wandb.init(
    project=WANDB_PROJECT,
    name=f"grpo-eval-{config['model']['name'].split('/')[-1]}",
    tags=["grpo-eval", "notebook-03"],
    config=config,
)
grpo_wandb = {
    "overall_pass_rate": grpo_metrics["overall_pass_rate"],
    **grpo_pass_k,
}
for cat, stats in grpo_metrics["per_category"].items():
    grpo_wandb[f"pass_rate/{cat}"] = stats["pass_rate"]
wandb.log(grpo_wandb)
wandb.finish()
print("\nMetriche post-GRPO loggate su wandb")

## 7. Confronto Baseline vs GRPO

In [ ]:
print(f"{'='*60}")
print(f"{'CONFRONTO':^60}")
print(f"{'='*60}")
print(f"{'Metrica':<30} {'Baseline':>12} {'GRPO':>12} {'Delta':>10}")
print(f"{'-'*60}")

b_rate = baseline_metrics["overall_pass_rate"]
g_rate = grpo_metrics["overall_pass_rate"]
delta = g_rate - b_rate
print(f"{'Overall Pass@1':<30} {b_rate:>12.4f} {g_rate:>12.4f} {delta:>+10.4f}")

# Per-category
all_cats = sorted(set(list(baseline_metrics["per_category"].keys()) + list(grpo_metrics["per_category"].keys())))
for cat in all_cats:
    b_cat = baseline_metrics["per_category"].get(cat, {}).get("pass_rate", 0.0)
    g_cat = grpo_metrics["per_category"].get(cat, {}).get("pass_rate", 0.0)
    d_cat = g_cat - b_cat
    print(f"  {cat:<28} {b_cat:>12.4f} {g_cat:>12.4f} {d_cat:>+10.4f}")

print(f"{'-'*60}")
if delta > 0:
    print(f"GRPO ha migliorato il pass rate di {delta:+.4f} ({delta/max(b_rate, 1e-6)*100:+.1f}%)")
elif delta < 0:
    print(f"GRPO ha peggiorato il pass rate di {delta:.4f}")
else:
    print("Nessun cambiamento")

In [ ]:
# Qualche esempio di completions post-GRPO
print("Esempi di completions post-GRPO:\n")
for i in [0, 1, 5, 10]:
    if i >= len(eval_ds):
        break
    comp = grpo_first[i]
    r = combined_reward(comp, task_type="json")
    print(f"--- [{difficulties[i]}] reward={r} ---")
    print(f"Prompt:     {eval_ds[i]['prompt'][:100]}...")
    print(f"Completion: {comp[:200]}")
    print()

In [ ]:
# Pulizia VRAM
del trainer
del model
torch.cuda.empty_cache()
print("VRAM liberata")

## Riepilogo

Questo notebook ha eseguito il flusso completo:

1. **Dataset** — 5000 samples JSON (simple/medium/hard)
2. **Baseline** — valutazione modello base → metriche su wandb
3. **GRPO Training** — addestramento con reward rule-based → loss/reward su wandb
4. **Post-GRPO** — valutazione modello trainato → confronto con baseline

Tutti i run sono su **wandb** sotto il progetto `grpo-strict-generation`.

**Comandi CLI equivalenti:**
```bash
# Baseline
uv run python -m src.evaluation.baseline_eval --config experiments/configs/baseline.yaml

# GRPO Training
uv run python -m src.training.grpo_train --config experiments/configs/grpo.yaml
```